In [0]:
from mlflow.deployments import get_deploy_client
from databricks.sdk import WorkspaceClient
import json
import uuid

import logging

In [0]:
def _convert_to_responses_format(messages):
    """Convert chat messages to ResponsesAgent API format."""
    input_messages = []
    for msg in messages:
        if msg["role"] == "user":
            input_messages.append({"role": "user", "content": msg["content"]})
        elif msg["role"] == "assistant":
            # Handle assistant messages with tool calls
            if msg.get("tool_calls"):
                # Add function calls
                for tool_call in msg["tool_calls"]:
                    input_messages.append({
                        "type": "function_call",
                        "id": tool_call["id"],
                        "call_id": tool_call["id"],
                        "name": tool_call["function"]["name"],
                        "arguments": tool_call["function"]["arguments"]
                    })
                # Add assistant message if it has content
                if msg.get("content"):
                    input_messages.append({
                        "type": "message",
                        "id": msg.get("id", str(uuid.uuid4())),
                        "content": [{"type": "output_text", "text": msg["content"]}],
                        "role": "assistant"
                    })
            else:
                # Regular assistant message
                input_messages.append({
                    "type": "message",
                    "id": msg.get("id", str(uuid.uuid4())),
                    "content": [{"type": "output_text", "text": msg["content"]}],
                    "role": "assistant"
                })
        elif msg["role"] == "tool":
            input_messages.append({
                "type": "function_call_output",
                "call_id": msg.get("tool_call_id"),
                "output": msg["content"]
            })
    return input_messages


In [0]:
async def _query_responses_endpoint_stream(endpoint_name: str, messages: list[dict[str, str]], return_traces: bool):
    """Stream responses from agent/v1/responses endpoints using MLflow deployments client."""
    client = get_deploy_client("databricks")
    
    input_messages = _convert_to_responses_format(messages)
    
    # Prepare input payload for ResponsesAgent
    inputs = {
        "input": input_messages,
        "context": {},
        "stream": True
    }
    if return_traces:
        inputs["databricks_options"] = {"return_trace": True}

    for event_data in await asyncio.to_thread(client.predict_stream, endpoint=endpoint_name, inputs=inputs):
        # Just yield the raw event data, let app.py handle the parsing
        yield event_data

In [0]:
import asyncio

TIMEOUT_SECONDS = 5

g = _query_responses_endpoint_stream(
  'agents_lucas_catalog-default-langgraph-mcp-responses-agent', 
  [{'role': 'user', 'content': 'qual o total de transações?'}], 
  return_traces=False
)

while True:
  try:
    i = await asyncio.wait_for(anext(g), timeout=TIMEOUT_SECONDS)
    print(i)
  except asyncio.TimeoutError:
    print("Timeout!")
    break
  except StopAsyncIteration:
    print("Done!")
    break

In [0]:
%pip install streamlit

In [0]:
from messages import UserMessage, AssistantResponse, render_message

In [0]:
render_message()

In [0]:
import streamlit as st

In [0]:
with st.chat_message("assistant") as m:
  st.write_stream(['Hello ', 'World'])
  print(m)

In [0]:
import time
import asyncio

def gen():
  for i in range(10):
    time.sleep(i)
    yield i

async def async_gen():
  for i in await asyncio.to_thread(gen):
    yield i

ag = async_gen()

async def async_process(ag):
  i = await anext(ag)
  print(i)

while True:
  try:
    await asyncio.wait_for(async_process(ag), timeout=2.1)
  except asyncio.TimeoutError:
    print("Timeout!")
    break
  except StopAsyncIteration:
    print("Done!")
    break

In [0]:
import asyncio

async def gen():
  for i in range(10):
    await asyncio.sleep(i)
    yield i

g = gen()

async def async_process(g):
  i = await anext(g)
  print(i)

while True:
  try:
    await asyncio.wait_for(async_process(g), timeout=5.1)
  except asyncio.TimeoutError:
    print("Timeout!")
    break
  except StopAsyncIteration:
    print("Done!")
    break

In [0]:
async for i in gen():
  await asyncio.wait_for(get_next(), timeout=3)
  print(i)

In [0]:
g = gen()
while True:
  i = None
  def get_next():
    i = next(g)
  await asyncio.wait_for(get_next(), timeout=3)
  print(i)

In [0]:
from multiprocessing import Process

TIMEOUT_SECONDS = 2

class TimeoutException(Exception):
  pass

g = gen()
while True:
  try:
    
    r = None
    def next_event():
      r = next(g)
    
    process = Process(target=next_event)
    process.start()
    process.join(timeout=TIMEOUT_SECONDS)
    
    if process.is_alive():
      process.terminate()
      raise TimeoutException
    
    print(r)
  except StopIteration:
    print("Done!")
    break
  except TimeoutException:
    print("Timeout!")
    break

In [0]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")

In [0]:
import mlflow

In [0]:
mlflow.environment_variables.MLFLOW_DEPLOYMENT_PREDICT_TIMEOUT = 5